# Progress Review 2 — Create Disruption Labels

This notebook creates disruption labels from Bronze weather data.

Primary rule from the project proposal:

label = 1 if wind_speed_kts > 25 OR precipitation_mm > 10

If the small PR2 sample has only one class, this notebook uses a temporary PR2 quantile fallback so Spark MLlib training can be demonstrated.

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")
        existing_sc.stop()
        print("Stopped existing SparkContext.")
    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")
    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
import pandas as pd
import s3fs

from deltalake import DeltaTable, write_deltalake

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("aviation-pr2-create-labels")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)

Spark version: 4.0.0
Spark app ID: app-20260425121234-0024


In [4]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT},
)

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_REGION": AWS_REGION,
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

In [5]:
bronze_delta_uri = "s3://lakehouse/bronze/weather_events_delta"

try:
    dt = DeltaTable(bronze_delta_uri, storage_options=storage_options)
    bronze_pdf = dt.to_pandas()
    print("Read Bronze from Delta:", bronze_delta_uri)

except Exception as exc:
    print("Delta read failed, falling back to Parquet.")
    print("Error:", repr(exc))

    bronze_parquet_path = "s3://lakehouse/bronze/weather_events_parquet/bronze_weather_events.parquet"
    with fs.open(bronze_parquet_path, "rb") as f:
        bronze_pdf = pd.read_parquet(f)

print("Bronze shape:", bronze_pdf.shape)
display(bronze_pdf.head())

if bronze_pdf.empty:
    raise RuntimeError("Bronze table is empty.")

Read Bronze from Delta: s3://lakehouse/bronze/weather_events_delta
Bronze shape: (72, 16)


,event_id,airport,timestamp_utc,event_date,latitude,longitude,temperature_c,wind_speed_kts,precipitation_mm,surface_pressure_pa,cape_j_kg,_kafka_topic,_kafka_partition,_kafka_offset,_ingested_at_utc,_source_object
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01T00:00:00Z,2022-01-01,40.75,286.25,9.424164,3.638606,0.005761,101271.765625,12.700439,weather.raw,0,0,2026-04-25T09:40:41.373487+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01T01:00:00Z,2022-01-01,40.75,286.25,9.465088,3.977354,0.010565,101288.007812,21.743164,weather.raw,0,1,2026-04-25T09:40:41.373644+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01T02:00:00Z,2022-01-01,40.75,286.25,9.402130,4.554652,0.012485,101208.484375,23.470459,weather.raw,0,2,2026-04-25T09:40:41.373665+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01T03:00:00Z,2022-01-01,40.75,286.25,9.392700,4.152695,0.001440,101199.937500,17.780762,weather.raw,0,3,2026-04-25T09:40:41.373677+00:00,kafka_weather_events/run_id=20260425T094031Z-d...
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01T04:00:00Z,2022-01-01,40.75,286.25,9.112518,4.710645,0.000959,101201.640625,14.834229,weather.raw,0,4,2026-04-25T09:40:41.373688+00:00,kafka_weather_events/run_id=20260425T094031Z-d...


In [6]:
bronze_df = spark.createDataFrame(bronze_pdf)

base_labeled_df = (
    bronze_df
    .withColumn("wind_speed_kts", F.col("wind_speed_kts").cast("double"))
    .withColumn("precipitation_mm", F.col("precipitation_mm").cast("double"))
    .withColumn("cape_j_kg", F.col("cape_j_kg").cast("double"))
    .withColumn(
        "weather_rule_label",
        F.when(
            (F.col("wind_speed_kts") > F.lit(25.0)) |
            (F.col("precipitation_mm") > F.lit(10.0)),
            F.lit(1.0),
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "risk_score_proxy",
        (F.col("wind_speed_kts") / F.lit(25.0)) +
        (F.col("precipitation_mm") / F.lit(10.0)) +
        (F.col("cape_j_kg") / F.lit(1000.0))
    )
)

print("Weather-rule label distribution:")
base_labeled_df.groupBy("weather_rule_label").count().show()

Weather-rule label distribution:
+------------------+-----+
|weather_rule_label|count|
+------------------+-----+
|               0.0|   72|
+------------------+-----+



In [7]:
class_count = base_labeled_df.select("weather_rule_label").distinct().count()

if class_count >= 2:
    labeled_df = (
        base_labeled_df
        .withColumn("label", F.col("weather_rule_label"))
        .withColumn("label_source", F.lit("WEATHER_RULE"))
    )

    print("Using WEATHER_RULE labels.")

else:
    quantile_value = base_labeled_df.approxQuantile("risk_score_proxy", [0.75], 0.01)[0]

    labeled_df = (
        base_labeled_df
        .withColumn(
            "label",
            F.when(F.col("risk_score_proxy") >= F.lit(quantile_value), F.lit(1.0))
            .otherwise(F.lit(0.0))
        )
        .withColumn("label_source", F.lit("PR2_QUANTILE_FALLBACK"))
    )

    print("Weather rule produced only one class.")
    print("Using PR2 quantile fallback threshold:", quantile_value)

print("Final label distribution:")
labeled_df.groupBy("label", "label_source").count().show()

Weather rule produced only one class.
Using PR2 quantile fallback threshold: 0.44497956085205076
Final label distribution:
+-----+--------------------+-----+
|label|        label_source|count|
+-----+--------------------+-----+
|  1.0|PR2_QUANTILE_FALL...|   19|
|  0.0|PR2_QUANTILE_FALL...|   53|
+-----+--------------------+-----+



In [8]:
labeled_pdf = labeled_df.toPandas()

silver_parquet_path = "s3://lakehouse/silver/weather_labeled_parquet/weather_labeled.parquet"

with fs.open(silver_parquet_path, "wb") as f:
    labeled_pdf.to_parquet(f, index=False)

print("Wrote Silver labeled Parquet:", silver_parquet_path)

silver_delta_uri = "s3://lakehouse/silver/weather_labeled_delta"

write_deltalake(
    silver_delta_uri,
    labeled_pdf,
    mode="overwrite",
    storage_options=storage_options,
)

print("Wrote Silver labeled Delta:", silver_delta_uri)
print("Rows:", len(labeled_pdf))

Wrote Silver labeled Parquet: s3://lakehouse/silver/weather_labeled_parquet/weather_labeled.parquet
Wrote Silver labeled Delta: s3://lakehouse/silver/weather_labeled_delta
Rows: 72


In [9]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
